In [ ]:
# §0-A ── Установка зависимостей (запустите первой)
import subprocess, sys
for pkg in ["tqdm", "ipywidgets"]:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg],
                   capture_output=True)
print("✓ Зависимости установлены")

In [ ]:
# §0-B ── Импорты
import math, random, warnings, itertools
from dataclasses import dataclass
from typing import List, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib import animation
from matplotlib.colors import Normalize
from IPython.display import HTML, display

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm.auto import tqdm

try:
    import ipywidgets as widgets
    HAS_WIDGETS = True
    print("✓ ipywidgets доступен — интерактивные виджеты активны")
except ImportError:
    HAS_WIDGETS = False
    print("⚠  ipywidgets недоступен — виджеты §6 запустятся без UI")

warnings.filterwarnings('ignore')
torch.set_default_dtype(torch.float32)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"✓ PyTorch {torch.__version__} | Устройство: {device}")

In [ ]:
# §0-C ── Тема оформления (переключаемая: светлая / тёмная)

# ┌─────────────────────────────────────────────────────┐
# │  Выберите тему: 'dark' или 'light'                  │
# └─────────────────────────────────────────────────────┘
THEME = 'light'   # <── меняйте здесь

# ── Палитры ───────────────────────────────────────────────────────────────────
_THEMES = {
    'dark': {
        'bg':      "#0d1117",
        'panel':   "#161b22",
        'text':    "#e6edf3",
        'grid':    "#30363d",
        'border':  "#21262d",
        'accent':  ["#58a6ff","#f78166","#3fb950","#d2a8ff",
                    "#ffa657","#79c0ff","#ff7b72","#a5d6ff"],
        'cmap_d':  "inferno",
        'cmap_t':  "plasma",
        'cmap_f':  "RdBu_r",
        'cmap_p':  "viridis",
        'cmap_e':  "magma",
        'leg_alpha': 0.12,
    },
    'light': {
        'bg':      "#ffffff",
        'panel':   "#f6f8fa",
        'text':    "#1c2128",
        'grid':    "#d0d7de",
        'border':  "#b0b8c1",
        'accent':  ["#0550ae","#c0392b","#1a7f37","#7c4dff",
                    "#d15a00","#0077b6","#e03e3e","#0284c7"],
        'cmap_d':  "YlOrRd",
        'cmap_t':  "cividis",
        'cmap_f':  "PuOr",
        'cmap_p':  "Blues",
        'cmap_e':  "hot",
        'leg_alpha': 0.85,
    },
}

def apply_theme(name: str):
    """Применить тему 'dark' или 'light' глобально."""
    assert name in _THEMES, f"Тема должна быть 'dark' или 'light', получено: '{name}'"
    T = _THEMES[name]

    plt.rcParams.update({
        "figure.facecolor":    T['bg'],
        "axes.facecolor":      T['panel'],
        "axes.edgecolor":      T['grid'],
        "axes.labelcolor":     T['text'],
        "text.color":          T['text'],
        "xtick.color":         T['text'],
        "ytick.color":         T['text'],
        "grid.color":          T['grid'],
        "grid.alpha":          0.35,
        "figure.dpi":          130,
        "font.family":         "monospace",
        "axes.titlesize":      13,
        "axes.labelsize":      11,
        "legend.framealpha":   T['leg_alpha'],
        "legend.edgecolor":    T['grid'],
        "legend.facecolor":    T['panel'],
        "lines.linewidth":     1.7,
        "axes.spines.top":     False,
        "axes.spines.right":   False,
        "savefig.facecolor":   T['bg'],
        "savefig.dpi":         150,
        "xtick.minor.visible": True,
        "ytick.minor.visible": True,
    })

    # Обновляем глобальные переменные цветов
    global DARK_BG, PANEL_BG, TEXT_C, GRID_C, BORDER_C, ACCENT
    global CMAP_D, CMAP_T, CMAP_F, CMAP_P, CMAP_E
    DARK_BG  = T['bg']
    PANEL_BG = T['panel']
    TEXT_C   = T['text']
    GRID_C   = T['grid']
    BORDER_C = T['border']
    ACCENT   = T['accent']
    CMAP_D   = T['cmap_d']
    CMAP_T   = T['cmap_t']
    CMAP_F   = T['cmap_f']
    CMAP_P   = T['cmap_p']
    CMAP_E   = T['cmap_e']

# Применяем выбранную тему
apply_theme(THEME)

# ── Вспомогательные функции рисования ────────────────────────────────────────
def styled_fig(w=10, h=6, title=None):
    fig = plt.figure(figsize=(w, h), facecolor=DARK_BG)
    if title:
        fig.suptitle(title, color=TEXT_C, fontsize=15, fontweight='bold', y=0.99)
    return fig

def add_colorbar(ax, im, label=""):
    from mpl_toolkits.axes_grid1 import make_axes_locatable
    div = make_axes_locatable(ax)
    cax = div.append_axes("right", size="4%", pad=0.07)
    cb  = plt.colorbar(im, cax=cax)
    cb.set_label(label, color=TEXT_C, fontsize=9)
    cb.ax.yaxis.set_tick_params(color=TEXT_C)
    plt.setp(cb.ax.yaxis.get_ticklabels(), color=TEXT_C)
    cax.set_facecolor(PANEL_BG)
    return cb

def kde_hist2d(x, bins=80, lim=4.5):
    x = x.detach().cpu().float().numpy() if hasattr(x, 'detach') else np.asarray(x)
    h, _, _ = np.histogram2d(x[:,0], x[:,1], bins=bins,
                              range=[[-lim,lim],[-lim,lim]], density=True)
    return h.T

# ── Превью текущей темы ───────────────────────────────────────────────────────
fig_prev, axes_prev = plt.subplots(1, 3, figsize=(14, 3.5), facecolor=DARK_BG)
fig_prev.suptitle(f"Тема: '{THEME}' — превью палитры и colormap'ов",
                  color=TEXT_C, fontsize=12, fontweight='bold')

# Полоса акцентных цветов
for i, col in enumerate(ACCENT):
    axes_prev[0].barh(i, 1, color=col, height=0.7)
    axes_prev[0].text(1.05, i, col, va='center', color=TEXT_C, fontsize=8)
axes_prev[0].set_xlim(0, 2.0); axes_prev[0].set_yticks(range(8))
axes_prev[0].set_yticklabels([f"ACCENT[{i}]" for i in range(8)], fontsize=8)
axes_prev[0].set_title("Акцентные цвета", color=TEXT_C)
axes_prev[0].set_facecolor(PANEL_BG)

# Colormap CMAP_D
grad = np.linspace(0, 1, 256).reshape(1, -1)
axes_prev[1].imshow(grad, aspect='auto', cmap=CMAP_D)
axes_prev[1].set_title(f"CMAP_D = '{CMAP_D}' (плотность)", color=TEXT_C)
axes_prev[1].set_yticks([])

# Colormap CMAP_F
axes_prev[2].imshow(grad, aspect='auto', cmap=CMAP_F)
axes_prev[2].set_title(f"CMAP_F = '{CMAP_F}' (поле дрейфа)", color=TEXT_C)
axes_prev[2].set_yticks([])

for ax in axes_prev: ax.set_facecolor(PANEL_BG)
plt.tight_layout(); plt.show()

print(f"✓ Тема '{THEME}' применена глобально")
print(f"  Фон:     {DARK_BG}  |  Панель: {PANEL_BG}")
print(f"  Текст:   {TEXT_C}  |  Сетка:  {GRID_C}")
print(f"  Чтобы переключить тему — измените THEME='light'/'dark' вверху ячейки и перезапустите её")

In [ ]:
# §0-D ── GlobalConfig — центральный управляющий объект
from dataclasses import dataclass

@dataclass
class GlobalConfig:
    # ── Симуляция ───────────────────────────────────────────
    n:             int   = 1024    # частиц
    n_steps:       int   = 120     # шагов SDE
    dt:            float = 1/120
    t_eps:         float = 1e-3

    # ── Sinkhorn OT ─────────────────────────────────────────
    epsilon:       float = 0.15
    sink_iters:    int   = 200
    sink_tol:      float = 1e-6

    # ── Нейросети ────────────────────────────────────────────
    hidden:        int   = 128
    epochs_diff:   int   = 200
    epochs_bridge: int   = 200
    lr:            float = 1e-3
    batch:         int   = 512

    # ── Визуализация ─────────────────────────────────────────
    lim:           float = 4.5
    bins:          int   = 80
    n_proj:        int   = 128
    n_traj:        int   = 100

    # ── VE-SDE ───────────────────────────────────────────────
    sigma_min:     float = 0.05
    sigma_max:     float = 15.0

CFG = GlobalConfig()

def set_seed(s=42):
    random.seed(s); np.random.seed(s); torch.manual_seed(s)
    if torch.cuda.is_available(): torch.cuda.manual_seed_all(s)

def to_np(t):
    return t.detach().cpu().float().numpy() if hasattr(t,'detach') else np.asarray(t)

set_seed(42)
print(f"✓ GlobalConfig: n={CFG.n}, steps={CFG.n_steps}, epsilon={CFG.epsilon}")
print(f"  Обучение: {CFG.epochs_diff} эпох диффузии · {CFG.epochs_bridge} эпох моста")
print("  Совет: для быстрого запуска установите epochs_diff=epochs_bridge=50")

In [ ]:
# §1-A ── Генераторы распределений
def generate_data(n: int, dist: str, seed: int = None) -> torch.Tensor:
    if seed is not None:
        rng = np.random.default_rng(seed)
    else:
        rng = np.random.default_rng()

    def _t(a): return torch.tensor(a, dtype=torch.float32)

    if dist == 'gaussian':
        return torch.randn(n, 2)

    elif dist == 'shifted_gaussian':
        return torch.randn(n, 2) * 0.4 + _t([-2.5, -2.5])

    elif dist == 'mixture_2':
        c  = rng.integers(0, 2, n)
        mu = np.array([[-2.5, 0.0], [2.5, 0.0]])
        return _t(rng.standard_normal((n, 2)) * 0.4 + mu[c])

    elif dist == 'mixture_8':
        angles = np.linspace(0, 2*np.pi, 8, endpoint=False)
        mu     = np.stack([np.cos(angles), np.sin(angles)], axis=1) * 3.0
        c      = rng.integers(0, 8, n)
        return _t(rng.standard_normal((n, 2)) * 0.18 + mu[c])

    elif dist == 'ring':
        theta = rng.uniform(0, 2*np.pi, n)
        r     = 2.5 + rng.standard_normal(n) * 0.18
        return _t(np.stack([r*np.cos(theta), r*np.sin(theta)], axis=1))

    elif dist == 'two_moons':
        h1 = n // 2
        t1 = rng.uniform(0, np.pi, h1)
        t2 = rng.uniform(np.pi, 2*np.pi, n-h1)
        x1 = np.stack([np.cos(t1)*2.5, np.sin(t1)*2.5-1.0], 1)
        x2 = np.stack([np.cos(t2)*2.5, np.sin(t2)*2.5+1.0], 1)
        return _t(np.concatenate([x1,x2],0) + rng.standard_normal((n,2))*0.15)

    elif dist == 'checkerboard':
        data = []
        while len(data) < n:
            pt = rng.uniform(-4, 4, (n, 2))
            ix = ((pt[:,0]+4)//1.6).astype(int) % 2
            iy = ((pt[:,1]+4)//1.6).astype(int) % 2
            data.extend(pt[(ix==iy)].tolist())
        return _t(np.array(data[:n]))

    elif dist == 'poisson_shells':
        n_c     = 5
        centers = rng.uniform(-2, 2, (n_c, 2))
        radii   = rng.uniform(0.8, 1.8, n_c)
        idx     = rng.integers(0, n_c, n)
        theta   = rng.uniform(0, 2*np.pi, n)
        r       = radii[idx] + rng.standard_normal(n)*0.12
        return _t(centers[idx] + np.stack([r*np.cos(theta), r*np.sin(theta)], 1))

    elif dist == 'uniform_square':
        return _t(rng.uniform(-3.5, 3.5, (n, 2)))

    elif dist == 'double_well':
        x = torch.randn(n, 2); dt_mc = 0.02
        for _ in range(2000):
            gx = 4*x[:,0]*(x[:,0]**2 - 2)
            gy = x[:,1]
            grad = torch.stack([gx, gy], dim=1)
            x = x - dt_mc*grad + math.sqrt(2*dt_mc)*torch.randn_like(x)
        return x.clamp(-5, 5)

    elif dist == 'lorenz_xz':
        sigma, rho, beta = 10.0, 28.0, 8.0/3
        xL, yL, zL = 1.0, 1.0, 1.0
        dt_l, warmup, traj = 0.005, 5000, []
        for i in range(warmup + n*4):
            dx = sigma*(yL-xL)*dt_l
            dy = (xL*(rho-zL)-yL)*dt_l
            dz = (xL*yL-beta*zL)*dt_l
            xL+=dx; yL+=dy; zL+=dz
            if i >= warmup and i % 4 == 0:
                traj.append([xL, zL])
        arr = np.array(traj[:n])
        arr = (arr - arr.mean(0)) / (arr.std(0) + 1e-8) * 1.8
        return _t(arr)

    else:
        raise ValueError(f"Unknown distribution: {dist}")

print("✓ Генераторы распределений загружены (11 типов)")

In [ ]:
# §1-B ── Визуализация всех распределений (scatter)
SHOW_DISTS = [
    ('gaussian',        'Гауссово N(0,I)'),
    ('shifted_gaussian','Смещённое N(μ,σ)'),
    ('mixture_2',       '2-компонентная смесь'),
    ('mixture_8',       '8-компонентная решётка'),
    ('ring',            'Кольцо (ротатор)'),
    ('two_moons',       'Две луны'),
    ('checkerboard',    'Шахматная доска'),
    ('poisson_shells',  'Пуассоновы оболочки'),
    ('uniform_square',  'Равномерное (квадрат)'),
    ('double_well',     'Двойная яма (метастаб.)'),
    ('lorenz_xz',       'Аттрактор Лоренца (x–z)'),
    ('mixture_8',       '8-GMM (второй экземпляр)'),
]

N_SCATTER = 3000   # точек на панель — меняйте по вкусу

fig, axes = plt.subplots(3, 4, figsize=(18, 13), facecolor=DARK_BG)
fig.suptitle("§1  Библиотека распределений — фазовое пространство",
             color=TEXT_C, fontsize=15, fontweight='bold', y=1.01)

for ax, (dname, dtitle) in zip(axes.flat, SHOW_DISTS):
    pts = to_np(generate_data(N_SCATTER, dname, seed=0))   # (N, 2)



    ax.scatter(pts[:, 0], pts[:, 1],
               c=ACCENT[0], cmap=CMAP_T,
               s=6, alpha=0.55, linewidths=0,
               rasterized=True)

    ax.set_xlim(-4.5, 4.5)
    ax.set_ylim(-4.5, 4.5)
    ax.set_aspect('equal')
    ax.set_title(dtitle, color=TEXT_C, fontsize=9.5, pad=4)
    ax.set_facecolor(PANEL_BG)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_edgecolor(GRID_C)

plt.tight_layout(pad=0.5)
plt.show()

---
## §2  Классический мост Шрёдингера

**Физическая постановка.**  Задача Шрёдингера (1931–32): найти наиболее вероятную
эволюцию ансамбля частиц с заданными краевыми условиями $\rho_0$ и $\rho_1$.

$$\pi^* = \arg\min_{\pi \in \Pi(\rho_0,\rho_1)} \mathrm{KL}(\pi \| \mathcal{W}_\varepsilon)$$

**Броуновский мост** интегрируется как SDE:
$$dX_t = \frac{x_1 - X_t}{1-t}\,dt + \sqrt{\varepsilon}\,dW_t, \quad t \in [0,1)$$

Дрейф $b_t = (x_1 - X_t)/(1-t)$ — **оптимальный управляющий протокол**,
минимизирующий стоимость транспорта при фиксированных краевых условиях.

In [ ]:
# §2-A ── Log-domain Sinkhorn (численно устойчивый)
@torch.no_grad()
def sinkhorn(x0, x1, epsilon=None, n_iters=None, tol=None, verbose=False):
    eps   = epsilon  or CFG.epsilon
    iters = n_iters  or CFG.sink_iters
    tol_  = tol      or CFG.sink_tol

    n, m  = x0.shape[0], x1.shape[0]
    a     = torch.full((n,), 1.0/n, device=x0.device)
    b     = torch.full((m,), 1.0/m, device=x1.device)
    C     = torch.cdist(x0, x1, p=2).pow(2)
    K_log = -C / eps
    log_u = torch.zeros(n, device=x0.device)
    log_v = torch.zeros(m, device=x1.device)
    log_a = a.log(); log_b = b.log()
    errs  = []

    it_rng = tqdm(range(iters), desc=f"Sinkhorn eps={eps:.3f}",
                  leave=False) if verbose else range(iters)
    for it in it_rng:
        log_u = log_a - torch.logsumexp(K_log + log_v[None,:], dim=1)
        log_v = log_b - torch.logsumexp(K_log.T + log_u[None,:], dim=1)
        if it % 20 == 0:
            pi_t = (log_u[:,None] + K_log + log_v[None,:]).exp()
            err  = (pi_t.sum(1) - a).abs().mean().item()
            errs.append(err)
            if err < tol_: break

    pi = (log_u[:,None] + K_log + log_v[None,:]).exp()
    pi = pi / pi.sum()
    return pi, C, log_u, log_v, errs

def coupling_stats(pi, C):
    cost     = (pi * C).sum().item()
    ent      = -(pi * pi.clamp_min(1e-30).log()).sum().item()
    sparsity = (pi < 1e-6).float().mean().item()
    return {"transport_cost": cost, "coupling_entropy": ent, "sparsity": sparsity}

print("✓ Log-domain Sinkhorn загружен")

In [ ]:
# §2-B ── Симуляция броуновского моста
@torch.no_grad()
def sample_couples(x0, x1, pi, n_pairs=None):
    n_pairs = n_pairs or x0.shape[0]
    flat = pi.reshape(-1); flat = flat / flat.sum()
    idx  = torch.multinomial(flat, n_pairs, replacement=True)
    i, j = idx // x1.shape[0], idx % x1.shape[0]
    return x0[i], x1[j]

@torch.no_grad()
def simulate_bridge(x0, x1, pi, epsilon=None, n_steps=None):
    eps  = epsilon  or CFG.epsilon
    T    = n_steps  or CFG.n_steps
    dt   = 1.0 / T
    n_p  = x0.shape[0]
    x0_s, x1_s = sample_couples(x0, x1, pi, n_p)
    x    = x0_s.clone()
    paths  = torch.empty(T+1, n_p, 2, device=x.device)
    drifts = torch.empty(T,   n_p, 2, device=x.device)
    paths[0] = x
    ts = torch.linspace(0, 1-CFG.t_eps, T, device=x.device)
    for k, t in enumerate(ts):
        b = (x1_s - x) / (1 - t).clamp_min(CFG.t_eps)
        drifts[k] = b
        x = x + b*dt + math.sqrt(eps*dt)*torch.randn_like(x)
        paths[k+1] = x
    return paths, (x0_s, x1_s), drifts

print("✓ Симулятор броуновского моста загружен")

In [ ]:
# §2-C ── Основной кейс: ring → mixture_8
set_seed(42)
SRC, TGT = 'ring', 'mixture_8'
x0_c = generate_data(CFG.n, SRC).to(device)
x1_c = generate_data(CFG.n, TGT).to(device)

print(f"Строим классический мост: {SRC} → {TGT}  (n={CFG.n})")
pi_c, C_c, _, _, errs_c = sinkhorn(x0_c, x1_c, verbose=True)
cs = coupling_stats(pi_c, C_c)
print(f"  Стоимость транспорта : {cs['transport_cost']:.4f}")
print(f"  Энтропия связывания  : {cs['coupling_entropy']:.4f}")
print(f"  Эффект. разреж-ть    : {cs['sparsity']*100:.1f}%")

paths_c, (x0s_c, x1s_c), drifts_c = simulate_bridge(x0_c, x1_c, pi_c)
print(f"  Форма путей: {tuple(paths_c.shape)}")

In [ ]:
# §2-D ── Визуализация: матрица π, сходимость, эволюция плотности, поле дрейфа
fig = styled_fig(22, 14, f"  Классический мост Шрёдингера: {SRC} → {TGT}")
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.38, wspace=0.30)

# ── Матрица связывания ─────────────────────────────────────────────────────
ax0 = fig.add_subplot(gs[0,0])
im0 = ax0.imshow(to_np(pi_c[:64,:64]), cmap=CMAP_T, aspect='auto')
ax0.set_title("Матрица связывания π\n(подматрица 64×64)", color=TEXT_C)
ax0.set_xlabel("j (цель)"); ax0.set_ylabel("i (источник)")
add_colorbar(ax0, im0, "π(i,j)")

# ── Сходимость Sinkhorn ────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs[0,1])
ax1.semilogy(errs_c, 'o-', color=ACCENT[0], lw=2, ms=5)
ax1.axhline(CFG.sink_tol, ls='--', color=ACCENT[1], lw=1.5, label=f"tol={CFG.sink_tol:.0e}")
ax1.set_xlabel("Итерации / 20"); ax1.set_ylabel("Ошибка КУ")
ax1.set_title("Сходимость Sinkhorn", color=TEXT_C)
ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# ── Эволюция плотности ─────────────────────────────────────────────────────
t_inds = [0, 30, 60, 90, 120]; t_lbls = ["t=0","t=0.25","t=0.5","t=0.75","t=1"]
for col, (tidx, tlbl) in enumerate(zip(t_inds[:2], t_lbls[:2])):
    ax = fig.add_subplot(gs[0, col+2])
    h  = kde_hist2d(paths_c[tidx].cpu(), bins=CFG.bins, lim=CFG.lim)
    im = ax.imshow(h, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
                   cmap=CMAP_D, aspect='equal', interpolation='bilinear')
    ax.set_title(tlbl, color=TEXT_C); ax.set_xticks([]); ax.set_yticks([])
    add_colorbar(ax, im, "p(x,t)")

for col, (tidx, tlbl) in enumerate(zip(t_inds[2:], t_lbls[2:])):
    ax = fig.add_subplot(gs[1, col+1] if col < 3 else gs[1, col])
    if col < 3:
        h  = kde_hist2d(paths_c[tidx].cpu(), bins=CFG.bins, lim=CFG.lim)
        im = ax.imshow(h, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
                       cmap=CMAP_D, aspect='equal', interpolation='bilinear')
        ax.set_title(tlbl, color=TEXT_C); ax.set_xticks([]); ax.set_yticks([])
        add_colorbar(ax, im, "p(x,t)")

# ── Траектории ────────────────────────────────────────────────────────────
ax_tr = fig.add_subplot(gs[1,0])
n_sh  = min(CFG.n_traj, paths_c.shape[1])
cols_tr = plt.cm.plasma(np.linspace(0.2, 0.9, n_sh))
for i in range(n_sh):
    tr = to_np(paths_c[:, i])
    ax_tr.plot(tr[:,0], tr[:,1], lw=0.45, alpha=0.3, color=cols_tr[i])
ax_tr.scatter(to_np(paths_c[0,:30,0]), to_np(paths_c[0,:30,1]),
              c=ACCENT[0], s=20, zorder=5, label="t=0")
ax_tr.scatter(to_np(paths_c[-1,:30,0]), to_np(paths_c[-1,:30,1]),
              c=ACCENT[1], s=20, zorder=5, label="t=1")
ax_tr.set_xlim(-CFG.lim,CFG.lim); ax_tr.set_ylim(-CFG.lim,CFG.lim)
ax_tr.set_aspect('equal'); ax_tr.set_title("Траектории частиц", color=TEXT_C)
ax_tr.legend(fontsize=8)

# ── Поле дрейфа на сетке (t=0.3) ─────────────────────────────────────────
ax_dr = fig.add_subplot(gs[1, 3])
k_dr  = int(0.3 * CFG.n_steps)
pts_t  = paths_c[k_dr].cpu(); drift_t = drifts_c[k_dr].cpu()
lim_d  = 4.0; ng = 18
gx_d   = torch.linspace(-lim_d, lim_d, ng)
gy_d   = torch.linspace(-lim_d, lim_d, ng)
GX_d, GY_d = torch.meshgrid(gx_d, gy_d, indexing='xy')
grid_d = torch.stack([GX_d.reshape(-1), GY_d.reshape(-1)], 1)
dists_d = torch.cdist(grid_d, pts_t)
nn_idx  = dists_d.topk(min(20, pts_t.shape[0]), largest=False).indices
dest_d  = torch.stack([drift_t[idx].mean(0) for idx in nn_idx])
U_d = dest_d[:,0].reshape(ng,ng).numpy()
V_d = dest_d[:,1].reshape(ng,ng).numpy()
spd_d = np.sqrt(U_d**2 + V_d**2)
h_bg = kde_hist2d(pts_t, bins=60, lim=CFG.lim)
ax_dr.imshow(h_bg, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
             cmap=CMAP_D, alpha=0.45, aspect='equal')
strm = ax_dr.streamplot(to_np(gx_d), to_np(gy_d), U_d.T, V_d.T,
                         color=spd_d.T, cmap=CMAP_F,
                         linewidth=1.3, density=1.3, arrowsize=1.4)
ax_dr.set_xlim(-lim_d,lim_d); ax_dr.set_ylim(-lim_d,lim_d)
ax_dr.set_title("Поле дрейфа b_t(x)  при t=0.3", color=TEXT_C)

for ax in fig.get_axes(): ax.set_facecolor(PANEL_BG)
plt.tight_layout(); plt.show()
print("✓ §2 завершён")

In [ ]:
# # §2-D ── Визуализация 1: Структура связей, метрики и поле дрейфа
# fig = styled_fig(16, 12, f"§2  Анализ моста Шрёдингера: {SRC} → {TGT}")
# gs  = gridspec.GridSpec(2, 2, figure=fig, hspace=0.38, wspace=0.30)

# # ── Матрица связывания ─────────────────────────────────────────────────────
# ax0 = fig.add_subplot(gs[0,0])
# # Отображаем подматрицу для читаемости (например, первые 64x64)
# im0 = ax0.imshow(to_np(pi_c[:64,:64]), cmap=CMAP_T, aspect='auto')
# ax0.set_title("Матрица связывания π\n(подматрица 64×64)", color=TEXT_C)
# ax0.set_xlabel("j (цель)"); ax0.set_ylabel("i (источник)")
# add_colorbar(ax0, im0, "π(i,j)")

# # ─ Сходимость Sinkhorn ────────────────────────────────────────────────────
# ax1 = fig.add_subplot(gs[0,1])
# ax1.semilogy(errs_c, 'o-', color=ACCENT[0], lw=2, ms=5)
# ax1.axhline(CFG.sink_tol, ls='--', color=ACCENT[1], lw=1.5,
#             label=f"tol={CFG.sink_tol:.0e}")
# ax1.set_xlabel("Итерации / 20"); ax1.set_ylabel("Ошибка КУ")
# ax1.set_title("Сходимость Sinkhorn", color=TEXT_C)
# ax1.legend(fontsize=9); ax1.grid(alpha=0.3)

# # ── Траектории частиц ─────────────────────────────────────────────────────
# ax_tr = fig.add_subplot(gs[1,0])
# n_sh  = min(CFG.n_traj, paths_c.shape[1])
# cols_tr = plt.cm.plasma(np.linspace(0.2, 0.9, n_sh))
# for i in range(n_sh):
#     tr = to_np(paths_c[:, i])
#     ax_tr.plot(tr[:,0], tr[:,1], lw=0.45, alpha=0.3, color=cols_tr[i])
# ax_tr.scatter(to_np(paths_c[0,:30,0]), to_np(paths_c[0,:30,1]),
#               c=ACCENT[0], s=20, zorder=5, label="t=0")
# ax_tr.scatter(to_np(paths_c[-1,:30,0]), to_np(paths_c[-1,:30,1]),
#               c=ACCENT[1], s=20, zorder=5, label="t=1")
# ax_tr.set_xlim(-CFG.lim,CFG.lim); ax_tr.set_ylim(-CFG.lim,CFG.lim)
# ax_tr.set_aspect('equal'); ax_tr.set_title("Траектории частиц", color=TEXT_C)
# ax_tr.legend(fontsize=8)

# # ─ Поле дрейфа на сетке (t=0.3) ─────────────────────────────────────────
# ax_dr = fig.add_subplot(gs[1, 1])
# k_dr  = int(0.3 * CFG.n_steps) # Индекс шага для t=0.3
# pts_t  = paths_c[k_dr].cpu(); drift_t = drifts_c[k_dr].cpu()
# lim_d  = 4.0; ng = 18

# # Создание сетки для визуализации векторного поля
# gx_d   = torch.linspace(-lim_d, lim_d, ng)
# gy_d   = torch.linspace(-lim_d, lim_d, ng)
# GX_d, GY_d = torch.meshgrid(gx_d, gy_d, indexing='xy')
# grid_d = torch.stack([GX_d.reshape(-1), GY_d.reshape(-1)], 1)

# # Вычисление вектора дрейфа через ближайшие точки (kNN)
# dists_d = torch.cdist(grid_d, pts_t)
# nn_idx  = dists_d.topk(min(20, pts_t.shape[0]), largest=False).indices
# dest_d  = torch.stack([drift_t[idx].mean(0) for idx in nn_idx])
# U_d = dest_d[:,0].reshape(ng,ng).numpy()
# V_d = dest_d[:,1].reshape(ng,ng).numpy()
# spd_d = np.sqrt(U_d**2 + V_d**2)

# # Фон плотности
# h_bg = kde_hist2d(pts_t, bins=60, lim=CFG.lim)
# ax_dr.imshow(h_bg, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
#              cmap=CMAP_D, alpha=0.45, aspect='equal')
# strm = ax_dr.streamplot(to_np(gx_d), to_np(gy_d), U_d.T, V_d.T,
#                          color=spd_d.T, cmap=CMAP_F,
#                          linewidth=1.3, density=1.3, arrowsize=1.4)
# ax_dr.set_xlim(-lim_d,lim_d); ax_dr.set_ylim(-lim_d,lim_d)
# ax_dr.set_title("Поле дрейфа b_t(x)  при t=0.3", color=TEXT_C)

# for ax in fig.get_axes(): ax.set_facecolor(PANEL_BG)
# plt.tight_layout(); plt.show()

In [ ]:
# # §2-D ── Визуализация 2: Эволюция плотности вероятности (2 строки)
# # Временные точки: 0%, 25%, 50%, 75%, 90% (t=1.0 заменен на t=0.9)
# t_fracs = [0.0, 0.25, 0.5, 0.75, 0.9]
# t_labels = ["t=0.0", "t=0.25", "t=0.5", "t=0.75", "t=0.9"]

# # Создаем фигуру с сеткой 2x3
# fig = plt.figure(figsize=(18, 10), facecolor=DARK_BG)
# gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.35, wspace=0.30)
# fig.suptitle(f"§2  Эволюция плотности: {SRC} → {TGT}",
#              color=TEXT_C, fontsize=14, fontweight='bold')

# # Создаем оси для 5 графиков
# axes = [
#     fig.add_subplot(gs[0, 0]),  # t=0.0
#     fig.add_subplot(gs[0, 1]),  # t=0.25
#     fig.add_subplot(gs[0, 2]),  # t=0.5
#     fig.add_subplot(gs[1, 0]),  # t=0.75
#     fig.add_subplot(gs[1, 1]),  # t=0.9
# ]

# for ax, frac, label in zip(axes, t_fracs, t_labels):
#     # Вычисляем индекс в пути
#     t_idx = int(frac * (paths_c.shape[0] - 1))

#     # Генерируем 2D гистограмму (KDE)
#     data = paths_c[t_idx].cpu()
#     h = kde_hist2d(data, bins=CFG.bins, lim=CFG.lim)

#     # Рисуем тепловую карту
#     im = ax.imshow(h, origin='lower', extent=[-CFG.lim, CFG.lim, -CFG.lim, CFG.lim],
#                    cmap=CMAP_D, aspect='equal', interpolation='bilinear')

#     # Оформление
#     ax.set_title(label, color=TEXT_C, fontsize=11)
#     ax.set_xticks([]); ax.set_yticks([])
#     ax.set_facecolor(PANEL_BG)

#     # Добавляем цветную шкалу к каждому графику
#     add_colorbar(ax, im, "p(x,t)")

# # Убираем лишнюю ячейку (нижний правый угол)
# fig.delaxes(fig.add_subplot(gs[1, 2]))

# plt.tight_layout()
# plt.show()

---
## §3  Диффузионная модель (VE-SDE + Score Matching)

**VE-SDE** — Variance-Exploding SDE, где дисперсия растёт со временем:
$$dX = \sqrt{\tfrac{d[\sigma^2(t)]}{dt}}\,dW_t, \quad \sigma(t) = \sigma_{\min}\!\left(\tfrac{\sigma_{\max}}{\sigma_{\min}}\right)^t$$

Генерация — **обращение времени** (Anderson 1982):
$$dX = -\tfrac{d[\sigma^2]}{dt}\,\nabla_x \log p_t(X)\,dt + \sqrt{\tfrac{d[\sigma^2]}{dt}}\,d\tilde{W}_t$$

**Loss (score matching):**
$$\mathcal{L} = \mathbb{E}_{t,x_0,\varepsilon}\!\left[\sigma^2(t)\left\|s_\theta(x_0+\sigma(t)\varepsilon,\,t) + \tfrac{\varepsilon}{\sigma(t)}\right\|^2\right]$$

In [ ]:
# §3-A ── Архитектуры нейросетей
class TimeEmbed(nn.Module):
    def __init__(self, dim, n_freqs=64):
        super().__init__()
        freqs = 2 ** torch.linspace(0, math.log2(1000), n_freqs)
        self.register_buffer('freqs', freqs)
        self.proj = nn.Sequential(
            nn.Linear(n_freqs*2, dim), nn.SiLU(), nn.Linear(dim, dim))

    def forward(self, t):
        t = 100.0 * t.view(-1, 1)
        emb = t * self.freqs[None,:]
        emb = torch.cat([torch.sin(emb), torch.cos(emb)], dim=-1)
        return self.proj(emb)

class ScoreNet(nn.Module):
    def __init__(self, d=2, hidden=None):
        super().__init__()
        h = hidden or CFG.hidden
        self.temb = TimeEmbed(h)
        self.net  = nn.Sequential(
            nn.Linear(d+h, h*2), nn.SiLU(),
            nn.Linear(h*2, h*2), nn.SiLU(),
            nn.Linear(h*2, h),   nn.SiLU(),
            nn.Linear(h, d))

    def forward(self, x, t):
        return self.net(torch.cat([x, self.temb(t)], dim=-1))

class BridgeDriftNet(nn.Module):
    def __init__(self, d=2, hidden=None):
        super().__init__()
        h = hidden or CFG.hidden
        self.temb = TimeEmbed(h)
        self.enc  = nn.Sequential(nn.Linear(d+h, h*2), nn.GELU(), nn.Linear(h*2, h))
        self.body = nn.Sequential(
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU(),
            nn.Linear(h, h), nn.LayerNorm(h), nn.GELU())
        self.head = nn.Linear(h, d)

    def forward(self, x, t):
        z = self.enc(torch.cat([x, self.temb(t)], dim=-1))
        return self.head(self.body(z) + z)  # residual connection

print("✓ ScoreNet и BridgeDriftNet загружены")
print(f"  ScoreNet:       {sum(p.numel() for p in ScoreNet().parameters()):,} параметров")
print(f"  BridgeDriftNet: {sum(p.numel() for p in BridgeDriftNet().parameters()):,} параметров")

In [ ]:
# §3-B ── Обучение и сэмплинг VE-SDE
# ┌──────────────────────────────────────────┐
# │  Целевое распределение — меняйте здесь   │
# └──────────────────────────────────────────┘
DIFF_TARGET = 'two_moons'

def sigma_t(t):
    return CFG.sigma_min * (CFG.sigma_max / CFG.sigma_min) ** t

def dsigma2_dt(t):
    return 2 * sigma_t(t)**2 * math.log(CFG.sigma_max / CFG.sigma_min)

def train_diffusion(dist, verbose=True):
    net = ScoreNet().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=CFG.lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.epochs_diff, eta_min=1e-5)
    data = generate_data(20000, dist)
    dl   = DataLoader(TensorDataset(data), batch_size=CFG.batch, shuffle=True)
    losses = []
    pbar = tqdm(range(CFG.epochs_diff), desc=f"Diffusion [{dist}]") if verbose else range(CFG.epochs_diff)
    for ep in pbar:
        ep_loss, n_b = 0.0, 0
        for (x0,) in dl:
            x0 = x0.to(device); bs = x0.shape[0]
            t   = torch.rand(bs, device=device)
            sig = torch.tensor([sigma_t(ti.item()) for ti in t],
                                device=device).view(-1, 1)
            eps  = torch.randn_like(x0)
            pred = net(x0 + sig*eps, t)
            loss = (sig.squeeze()**2 *
                    (pred + eps/sig.squeeze()[:,None].clamp_min(1e-8)).pow(2).sum(-1)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); n_b += 1
        sch.step(); losses.append(ep_loss / n_b)
        if verbose:
            pbar.set_postfix(loss=f"{losses[-1]:.4f}", lr=f"{sch.get_last_lr()[0]:.1e}")
    return net, losses

@torch.no_grad()
def sample_diffusion_sde(net, n, n_steps=400, t_start=1.0):
    """Ланжевеновский SDE-сэмплинг с сохранением траекторий."""
    net.eval()
    ts    = torch.linspace(t_start, 0.002, n_steps, device=device)
    x     = sigma_t(t_start) * torch.randn(n, 2, device=device)
    paths = [x.clone()]
    for i in range(n_steps - 1):
        t_v = ts[i].item(); dt = abs((ts[i+1] - ts[i]).item())
        score = net(x, torch.full((n,), t_v, device=device))
        ds2   = dsigma2_dt(t_v)
        drift = 0.5 * ds2 * score
        x = x + drift*dt + math.sqrt(ds2*dt) * torch.randn_like(x)
        paths.append(x.clone())
    return x, torch.stack(paths)   # (final, T×N×2)

@torch.no_grad()
def sample_diffusion_ode(net, n, n_steps=400, t_start=1.0):
    """Probability-flow ODE (детерминированный)."""
    net.eval()
    ts = torch.linspace(t_start, 0.002, n_steps, device=device)
    x  = sigma_t(t_start) * torch.randn(n, 2, device=device)
    for i in range(n_steps - 1):
        t_v = ts[i].item(); dt = abs((ts[i+1] - ts[i]).item())
        score = net(x, torch.full((n,), t_v, device=device))
        x = x + 0.5 * dsigma2_dt(t_v) * score * dt
    return x

# ── Обучение ──────────────────────────────────────────────────────────────────
print(f"Обучение ScoreNet (target: {DIFF_TARGET})...")
score_net, diff_losses = train_diffusion(DIFF_TARGET)

# ── Кривая потерь (вместо матрицы π) ─────────────────────────────────────────
fig, ax_loss = plt.subplots(1, 1, figsize=(8, 4))
ax_loss.plot(diff_losses, color=ACCENT[0], lw=1.5)
ax_loss.set_xlabel("Эпоха"); ax_loss.set_ylabel("Loss")
ax_loss.set_title("Кривая обучения\n(score matching)", color=TEXT_C)
ax_loss.set_yscale('log'); ax_loss.grid(alpha=0.3)


print(f"✓ §3 завершён  |  target: {DIFF_TARGET}  |  loss = {diff_losses[-1]:.4f}")

In [ ]:
# §3-B ── Cэмплинг VE-SDE

# ── Сэмплинг: SDE с путями + ODE финал ───────────────────────────────────────
N_SAMP = CFG.n
_, paths_diff = sample_diffusion_sde(score_net, N_SAMP, n_steps=400)
samples_diff  = sample_diffusion_ode(score_net, N_SAMP, n_steps=400)
T_diff        = paths_diff.shape[0]   # число шагов

# ── Визуализация (макет совпадает с §2-D) ─────────────────────────────────────
fig = styled_fig(22, 14, f"§ Диффузионная модель VE-SDE: → {DIFF_TARGET}")
gs  = gridspec.GridSpec(2, 4, figure=fig, hspace=0.38, wspace=0.30)

# ── Эволюция плотности: t=1 (шум) → t=0 (данные), 2 кадра в первом ряду ─────
# Индексы в paths_diff: 0 = чистый шум (t=1), -1 = сгенерированное (t≈0)
# Показываем обратный процесс — генерацию
t_fracs  = [0.0, 0.25]          # доля пройденного пути (0 = шум, 1 = данные)
t_labels = ["t=1.0  (шум)", "t=0.75"]
for col, (frac, lbl) in enumerate(zip(t_fracs, t_labels)):
    tidx = int(frac * (T_diff - 1))
    pts  = paths_diff[tidx].cpu()
    ax   = fig.add_subplot(gs[0, col + 2])
    ax.scatter(to_np(pts[:, 0]), to_np(pts[:, 1]),
               c=ACCENT[0], s=4, alpha=0.35, linewidths=0, rasterized=True)
    ax.set_xlim(-CFG.lim, CFG.lim); ax.set_ylim(-CFG.lim, CFG.lim)
    ax.set_aspect('equal'); ax.set_title(lbl, color=TEXT_C)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(PANEL_BG)

t_fracs2  = [0.5, 0.75, 1.0]
t_labels2 = ["t=0.5", "t=0.25", "t=0  (ρ₁)"]
for col, (frac, lbl) in enumerate(zip(t_fracs2, t_labels2)):
    tidx = int(frac * (T_diff - 1))
    pts  = paths_diff[tidx].cpu() if frac < 1.0 else samples_diff.cpu()
    ax   = fig.add_subplot(gs[1, col + 1])
    ax.scatter(to_np(pts[:, 0]), to_np(pts[:, 1]),
               c=ACCENT[1] if frac == 1.0 else ACCENT[0],
               s=4, alpha=0.35, linewidths=0, rasterized=True)
    ax.set_xlim(-CFG.lim, CFG.lim); ax.set_ylim(-CFG.lim, CFG.lim)
    ax.set_aspect('equal'); ax.set_title(lbl, color=TEXT_C)
    ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(PANEL_BG)


# ── Поле score на сетке (t = 0.5) ────────────────────────────────────────────
ax_sc = fig.add_subplot(gs[0, 1])
t_sc  = 0.5
lim_s = 4.0; ng = 18
gx_s  = torch.linspace(-lim_s, lim_s, ng)
gy_s  = torch.linspace(-lim_s, lim_s, ng)
GX_s, GY_s = torch.meshgrid(gx_s, gy_s, indexing='xy')
grid_s = torch.stack([GX_s.reshape(-1), GY_s.reshape(-1)], 1).to(device)
t_vec  = torch.full((grid_s.shape[0],), t_sc, device=device)
with torch.no_grad():
    score_field = score_net(grid_s, t_vec).cpu()
U_s = score_field[:, 0].reshape(ng, ng).numpy()
V_s = score_field[:, 1].reshape(ng, ng).numpy()
spd_s = np.sqrt(U_s**2 + V_s**2)

# Фон — промежуточное облако точек
tidx_bg = int(0.5 * (T_diff - 1))
pts_bg  = to_np(paths_diff[tidx_bg].cpu())
ax_sc.scatter(pts_bg[:, 0], pts_bg[:, 1],
              c=ACCENT[0], s=4, alpha=0.25, linewidths=0, rasterized=True)
strm = ax_sc.streamplot(to_np(gx_s), to_np(gy_s), U_s.T, V_s.T,
                         color=spd_s.T, cmap=CMAP_F,
                         linewidth=1.3, density=1.3, arrowsize=1.4)
ax_sc.set_xlim(-lim_s, lim_s); ax_sc.set_ylim(-lim_s, lim_s)
ax_sc.set_title(f"Score field s_θ(x,t)\nпри t={t_sc}", color=TEXT_C)
ax_sc.set_facecolor(PANEL_BG)

for ax in fig.get_axes():
    ax.set_facecolor(PANEL_BG)

plt.tight_layout()
plt.show()

In [ ]:
# §3-B ── Обучение и сэмплинг VE-SDE
def sigma_t(t):
    return CFG.sigma_min * (CFG.sigma_max / CFG.sigma_min) ** t

def dsigma2_dt(t):
    return 2 * sigma_t(t)**2 * math.log(CFG.sigma_max / CFG.sigma_min)

def train_diffusion(dist='poisson_shells', verbose=True):
    net = ScoreNet().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=CFG.lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.epochs_diff, eta_min=1e-5)
    data = generate_data(20000, dist)
    dl   = DataLoader(TensorDataset(data), batch_size=CFG.batch, shuffle=True)
    losses = []
    pbar = tqdm(range(CFG.epochs_diff), desc="Diffusion") if verbose else range(CFG.epochs_diff)
    for ep in pbar:
        ep_loss, n_b = 0.0, 0
        for (x0,) in dl:
            x0 = x0.to(device); bs = x0.shape[0]
            t  = torch.rand(bs, device=device)
            sig = torch.tensor([sigma_t(ti.item()) for ti in t], device=device).view(-1,1)
            eps = torch.randn_like(x0)
            pred = net(x0 + sig*eps, t)
            loss = (sig.squeeze()**2 * (pred + eps/sig.squeeze()[:,None].clamp_min(1e-8)).pow(2).sum(-1)).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); n_b += 1
        sch.step(); losses.append(ep_loss/n_b)
        if verbose: pbar.set_postfix(loss=f"{losses[-1]:.4f}", lr=f"{sch.get_last_lr()[0]:.1e}")
    return net, losses

@torch.no_grad()
def sample_diffusion_ode(net, n, n_steps=400, t_start=1.0):
    net.eval()
    ts = torch.linspace(t_start, 0.002, n_steps, device=device)
    x  = sigma_t(t_start) * torch.randn(n, 2, device=device)
    for i in range(n_steps-1):
        t_v = ts[i].item(); dt = abs((ts[i+1]-ts[i]).item())
        score = net(x, torch.full((n,), t_v, device=device))
        x = x + 0.5 * dsigma2_dt(t_v) * score * dt
    return x

print("Обучение ScoreNet (target: mixture_8)...")
score_net, diff_losses = train_diffusion('mixture_8')
samples_diff = sample_diffusion_ode(score_net, 2000)

# ── Визуализация ────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=DARK_BG)
fig.suptitle(" Диффузионная модель VE-SDE (target: mixture_8)",
             color=TEXT_C, fontsize=14, fontweight='bold')

axes[0].plot(diff_losses, color=ACCENT[0], lw=1.5)
axes[0].set_xlabel("Эпоха"); axes[0].set_ylabel("Loss")
axes[0].set_title("Кривая обучения", color=TEXT_C)
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3)

for col, (xdata, title) in enumerate([
    (generate_data(2000,'mixture_8'), "Целевое ρ₁ (mixture_8)"),
    (samples_diff.cpu(),              "Сгенерированные сэмплы (ODE)")
]):
    h  = kde_hist2d(xdata, bins=CFG.bins, lim=CFG.lim)
    im = axes[col+1].imshow(h, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
                             cmap=CMAP_D, aspect='equal', interpolation='bilinear')
    axes[col+1].set_title(title, color=TEXT_C)
    add_colorbar(axes[col+1], im)

for ax in axes: ax.set_facecolor(PANEL_BG)
plt.tight_layout(); plt.show()
print(f"✓ §3 завершён  |  финальный loss = {diff_losses[-1]:.4f}")

---
## §4  Нейронный мост Шрёдингера (Flow Matching)

**Flow Matching** — параметризовать поле скоростей $u_\theta(x, t)$, транспортирующее $\rho_0 \to \rho_1$:

$$\mathcal{L}_{\rm FM} = \mathbb{E}_{t,\,(x_0,x_1)\sim\pi^*}\!\left[\|u_\theta(x_t, t) - (x_1 - x_0)\|^2\right]$$

где $x_t = (1-t)x_0 + t x_1 + \sqrt{t(1-t)\,\varepsilon}\,\xi$.

**Soft coupling:** пары $(x_0, x_1)$ берутся из **матрицы Sinkhorn** $\pi^*$ — это
отличает мост от независимого flow matching и гарантирует оптимальность транспорта.

In [ ]:
# §4-A ── Обучение нейронного моста
def train_neural_bridge(src_dist, tgt_dist, verbose=True):
    net = BridgeDriftNet().to(device)
    opt = torch.optim.Adam(net.parameters(), lr=CFG.lr)
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, CFG.epochs_bridge, eta_min=2e-5)

    x0_all = generate_data(20000, src_dist).to(device)
    x1_all = generate_data(20000, tgt_dist).to(device)
    dl0 = DataLoader(TensorDataset(x0_all), batch_size=CFG.batch, shuffle=True, drop_last=True)
    dl1 = DataLoader(TensorDataset(x1_all), batch_size=CFG.batch, shuffle=True, drop_last=True)
    losses = []
    pbar = tqdm(range(CFG.epochs_bridge), desc=f"Bridge {src_dist}→{tgt_dist}") if verbose else range(CFG.epochs_bridge)

    for ep in pbar:
        it0, it1 = iter(dl0), iter(dl1)
        ep_loss, n_b = 0.0, 0
        for _ in range(len(dl0)):
            x0 = next(it0)[0]; x1 = next(it1)[0]
            B  = min(x0.shape[0], x1.shape[0]); x0, x1 = x0[:B], x1[:B]
            with torch.no_grad():
                pi_b, _, _, _, _ = sinkhorn(x0, x1, epsilon=0.08, n_iters=50)
                flat = pi_b.reshape(-1); flat = flat/flat.sum()
                idx_p = torch.multinomial(flat, B, replacement=True)
                i_p = idx_p // B; j_p = idx_p % B
                x0_p = x0[i_p]; x1_p = x1[j_p]
            t   = torch.rand(B, 1, device=device)
            xt  = (1-t)*x0_p + t*x1_p + (t*(1-t)*CFG.epsilon).sqrt()*torch.randn_like(x0_p)
            pred = net(xt, t.squeeze())
            loss = (pred - (x1_p - x0_p)).pow(2).sum(-1).mean()
            opt.zero_grad(); loss.backward(); opt.step()
            ep_loss += loss.item(); n_b += 1
        sch.step(); losses.append(ep_loss/n_b)
        if verbose: pbar.set_postfix(loss=f"{losses[-1]:.4f}")
    return net, losses

@torch.no_grad()
def sample_bridge_ode(net, src_dist, n=None, n_steps=120):
    """Сэмплинг нейронного моста с остановкой на t=0.9 для избежания численных артефактов."""
    net.eval()
    n  = n or CFG.n
    x  = generate_data(n, src_dist).to(device)
    # Изменено: временная сетка от 0 до 0.9 вместо 1.0
    ts = torch.linspace(0, 0.9, n_steps+1, device=device)
    paths = [x.clone()]
    for i in range(n_steps):
        t_v = ts[i].item(); dt = abs(ts[i+1]-ts[i]).item()
        u   = net(x, torch.full((n,), t_v, device=device))
        x   = x + u*dt + math.sqrt(CFG.epsilon*dt)*torch.randn_like(x)
        paths.append(x.clone())
    return x, torch.stack(paths)

print("Обучение нейронного моста (ring → mixture_8)...")
bridge_net, bridge_losses = train_neural_bridge('ring', 'mixture_8')
samples_bridge, paths_nb  = sample_bridge_ode(bridge_net, 'ring', n=CFG.n)

fig, axes = plt.subplots(1, 4, figsize=(22, 5), facecolor=DARK_BG)
fig.suptitle("§4  Нейронный мост (Flow Matching): ring → mixture_8",
             color=TEXT_C, fontsize=14, fontweight='bold')

axes[0].plot(bridge_losses, color=ACCENT[2], lw=1.5)
axes[0].set_xlabel("Эпоха"); axes[0].set_ylabel("FM Loss")
axes[0].set_title("Кривая обучения", color=TEXT_C)
axes[0].set_yscale('log'); axes[0].grid(alpha=0.3)

# Изменено: индексы и подписи для t=0, t=0.5, t=0.9
# При n_steps=120 и t_max=0.9: dt=0.0075, поэтому:
# t=0.0 → idx=0, t=0.5 → idx≈67, t=0.9 → idx=120
for col, (tidx, tlbl) in enumerate(zip([0, 67, 120], ["t=0.0", "t=0.5", "t=0.9"])):
    h  = kde_hist2d(paths_nb[tidx].cpu(), bins=CFG.bins, lim=CFG.lim)
    im = axes[col+1].imshow(h, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
                             cmap=CMAP_D, aspect='equal', interpolation='bilinear')
    axes[col+1].set_title(f"Нейр. мост  {tlbl}", color=TEXT_C)
    add_colorbar(axes[col+1], im, "p(x,t)")

for ax in axes: ax.set_facecolor(PANEL_BG)
plt.tight_layout(); plt.show()
print(f"✓ §4 завершён  |  финальный loss = {bridge_losses[-1]:.4f}")

---
## §5  Неравновесная термодинамика моста Шрёдингера

| Термодинамика | Мост Шрёдингера |
|---|---|
| Свободная энергия $F = U - TS$ | $\langle C\rangle - \varepsilon H[\pi]$ |
| Производство энтропии $\sigma$ | $\frac{1}{\varepsilon}\int_0^1\mathbb{E}\|b_t\|^2\,dt$ |
| Стрела времени | $D_\mathrm{KL}(P_\to\|P_\leftarrow) = \sigma > 0$ |
| Неравенство Жарзинского | $\langle e^{-W/\varepsilon}\rangle = e^{-\Delta F/\varepsilon}$ |
| Равновесие | $\varepsilon \to \infty$ |

**Производство энтропии** — центральный термодинамический инвариант моста:
$$\sigma[X_\cdot] = \frac{1}{\varepsilon}\int_0^1 \|b_t(X_t)\|^2\,dt$$

In [ ]:
# §5-A ── Термодинамические функционалы
def gaussian_entropy(x) -> float:
    xn  = to_np(x)
    cov = np.cov(xn.T) + 1e-6*np.eye(2)
    return 0.5 * np.log(np.linalg.det(2*np.pi*np.e*cov))

def gaussian_kl(x, y) -> float:
    xn, yn   = to_np(x), to_np(y)
    mu_p, cov_p = xn.mean(0), np.cov(xn.T)+1e-6*np.eye(2)
    mu_q, cov_q = yn.mean(0), np.cov(yn.T)+1e-6*np.eye(2)
    inv_q = np.linalg.inv(cov_q); diff = mu_p - mu_q
    return 0.5*(np.trace(inv_q@cov_p) + diff@inv_q@diff - 2
                + np.log(np.linalg.det(cov_q)/(np.linalg.det(cov_p)+1e-12)+1e-12))

def entropy_production(drifts, dt, epsilon):
    ep_density = drifts.pow(2).sum(-1).mean(1) * dt / epsilon
    return ep_density.sum().item(), to_np(ep_density)

def path_work(paths, drifts, epsilon, dt):
    T  = drifts.shape[0]
    dx = paths[1:T+1] - paths[:T]
    W  = (-(drifts*dx).sum(-1) + 0.5*drifts.pow(2).sum(-1)*dt).sum(0)
    return W / epsilon

def time_arrow(paths, epsilon, dt) -> float:
    fv = (paths[1:] - paths[:-1]) / dt
    rv = (paths.flip(0)[1:] - paths.flip(0)[:-1]) / dt
    return abs(fv.pow(2).sum(-1).mean().item() - rv.pow(2).sum(-1).mean().item()) * dt / epsilon

print("✓ Термодинамические функционалы определены")

In [ ]:
# §5-B ── Вычисление и отчёт
N_TS = 10
ts_th = np.linspace(0, 1, N_TS)
entropies, kl_fwd, kl_bwd = [], [], []
x1_ref_cpu = x1_c.cpu()

for tv in ts_th:
    k = min(int(tv * CFG.n_steps), paths_c.shape[0]-1)
    x_k = paths_c[k].cpu()
    entropies.append(gaussian_entropy(x_k))
    kl_fwd.append(gaussian_kl(x0_c.cpu(), x_k))
    kl_bwd.append(gaussian_kl(x_k, x1_ref_cpu))

sigma_total, sigma_density = entropy_production(drifts_c.cpu(), CFG.dt, CFG.epsilon)
W_paths  = path_work(paths_c.cpu(), drifts_c.cpu(), CFG.epsilon, CFG.dt)
jar_lhs  = torch.exp(-W_paths).mean().item()
jar_rhs  = math.exp(-(entropies[0]-entropies[-1]))
ta       = time_arrow(paths_c.cpu(), CFG.epsilon, CFG.dt)

print("╔" + "═"*50 + "╗")
print("║     ТЕРМОДИНАМИЧЕСКИЙ ОТЧЁТ — МОСТ ШРЁДИНГЕРА     ║")
print("╠" + "═"*50 + "╣")
print(f"║  H[ρ₀]            = {entropies[0]:>8.4f}  nats              ║")
print(f"║  H[ρ₁]            = {entropies[-1]:>8.4f}  nats              ║")
print(f"║  ΔH = H₁ - H₀     = {entropies[-1]-entropies[0]:>8.4f}  nats              ║")
print(f"║  σ (произв. энтр.)= {sigma_total:>8.4f}  nats              ║")
print(f"║  Жарзинский LHS   = {jar_lhs:>8.4f}                    ║")
print(f"║  Жарзинский RHS   = {jar_rhs:>8.4f}                    ║")
print(f"║  Стрела времени   = {ta:>8.4f}                    ║")
print("╚" + "═"*50 + "╝")

In [ ]:
# §5-C ── Термодинамические временны́е ряды
ts_plot = np.linspace(0, 1, len(sigma_density))
drift_mag = to_np(drifts_c.cpu().norm(dim=-1).mean(1))

fig = styled_fig(22, 13, "§5  Неравновесная термодинамика моста Шрёдингера")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.42, wspace=0.38)

def thermo_ax(ax, xs, ys, color, ylabel, title, fill=True):
    ax.plot(xs, ys, 'o-', color=color, lw=2.2, ms=6)
    if fill: ax.fill_between(xs, min(ys)*0.95, ys, alpha=0.18, color=color)
    ax.set_xlabel("t"); ax.set_ylabel(ylabel)
    ax.set_title(title, color=TEXT_C)
    ax.grid(alpha=0.3); ax.set_facecolor(PANEL_BG)

thermo_ax(fig.add_subplot(gs[0,0]), ts_th, entropies,
          ACCENT[0], "H[ρ_t]  (nats)", "Дифференциальная энтропия H(t)")

ax_kl = fig.add_subplot(gs[0,1])
ax_kl.plot(ts_th, kl_fwd, 'o-', color=ACCENT[0], lw=2, ms=6, label="KL(ρ₀‖ρ_t)")
ax_kl.plot(ts_th, kl_bwd, 's-', color=ACCENT[1], lw=2, ms=6, label="KL(ρ_t‖ρ₁)")
ax_kl.fill_between(ts_th, kl_fwd, kl_bwd, alpha=0.1, color=ACCENT[3])
ax_kl.text(0.5, max(max(kl_fwd),max(kl_bwd))*0.92,
           f"Стрела вр.: {ta:.3f}", ha='center', color=ACCENT[3], fontsize=9)
ax_kl.set_xlabel("t"); ax_kl.set_ylabel("KL (nats)")
ax_kl.set_title("KL-дивергенции и стрела времени", color=TEXT_C)
ax_kl.legend(fontsize=9); ax_kl.grid(alpha=0.3); ax_kl.set_facecolor(PANEL_BG)

thermo_ax(fig.add_subplot(gs[0,2]), ts_plot, sigma_density,
          ACCENT[2], "dσ/dt", f"Производство энтропии σ(t)\nΣ={sigma_total:.3f} nats")

thermo_ax(fig.add_subplot(gs[1,0]),
          np.linspace(0,1,len(drift_mag)), drift_mag,
          ACCENT[4], "E[‖b_t‖]", "Средняя скорость поля дрейфа")

# Аналог Жарзинского
ax_j = fig.add_subplot(gs[1,1])
w_np = to_np(W_paths)
ax_j.hist(w_np, bins=50, density=True, color=ACCENT[5], alpha=0.7, label="P(W/ε)")
ax_j.axvline(w_np.mean(), color=ACCENT[1], lw=2, label=f"⟨W/ε⟩={w_np.mean():.3f}")
ax_j.axvline(entropies[-1]-entropies[0], color=ACCENT[0], lw=2, ls='--',
             label=f"ΔH={entropies[-1]-entropies[0]:.3f}")
ax_j.set_xlabel("W/ε"); ax_j.set_ylabel("Плотность")
ax_j.set_title(f"Распределение работы (Жарзинский)\n"
               f"⟨e^(-W/ε)⟩={jar_lhs:.3f}  vs  e^(-ΔH)={jar_rhs:.3f}", color=TEXT_C, fontsize=10)
ax_j.legend(fontsize=8); ax_j.set_facecolor(PANEL_BG)

# Кинетическая энергия управления
en_dens = to_np(drifts_c.cpu().pow(2).sum(-1).mean(1))
thermo_ax(fig.add_subplot(gs[1,2]), ts_plot, en_dens,
          ACCENT[6], "E[‖b_t‖²]", "Кинетическая энергия управления")

plt.tight_layout(); plt.show()
print("✓ §5 завершён")

---
## §6  Фазовые переходы: ε-скан

Параметр $\varepsilon$ — **обратная температура** $\beta^{-1}$:

| $\varepsilon \to 0$ | $\varepsilon \to \infty$ |
|---|---|
| Детерминированный OT | Свободная диффузия |
| Максимальная необратимость | Термодинамическое равновесие |
| $\sigma \to \infty$ | $\sigma \to 0$ |

**Гипотеза:** при переходе между топологически различными $\rho_0, \rho_1$ существует
критический $\varepsilon_c$, при котором поведение Sinkhorn и структура $\pi^*$
качественно меняются — аналог фазового перехода.

In [ ]:
# §6-A ── ε-скан и сбор диагностик
EPSILONS = [0.01, 0.03, 0.06, 0.10, 0.18, 0.35, 0.70, 1.5, 3.0, 6.0]
N_SCAN   = 512
x0_sc = generate_data(N_SCAN, 'ring').to(device)
x1_sc = generate_data(N_SCAN, 'mixture_8').to(device)

scan_rows = []
print("ε-скан: ring → mixture_8  ...")
for eps in tqdm(EPSILONS, desc="epsilon scan"):
    pi_e, C_e, _, _, errs_e = sinkhorn(x0_sc, x1_sc, epsilon=eps, n_iters=300)
    paths_e, _, drifts_e    = simulate_bridge(x0_sc, x1_sc, pi_e, epsilon=eps, n_steps=80)
    sig_e, _     = entropy_production(drifts_e.cpu(), 1/80, eps)
    trans_cost   = (pi_e * C_e).sum().item()
    coup_ent     = -(pi_e * pi_e.clamp_min(1e-30).log()).sum().item()
    n_iters_cv   = len(errs_e) * 20
    h_mid        = gaussian_entropy(paths_e[40].cpu())
    scan_rows.append(dict(epsilon=eps, sigma=sig_e, cost=trans_cost,
                          entropy=coup_ent, n_iters=n_iters_cv, h_mid=h_mid))

df_scan = pd.DataFrame(scan_rows)
print(df_scan.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
# §6-B ── Фазовые диаграммы
fig = styled_fig(22, 14, "§6  Фазовые переходы: ε-скан (ring → mixture_8)")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.38)

def eps_ax(ax, y_arr, ylabel, color, title):
    ax.semilogx(df_scan.epsilon, y_arr, 'o-', color=color, lw=2.2, ms=9,
                markeredgecolor=DARK_BG, markeredgewidth=1.5)
    ax.fill_between(df_scan.epsilon, min(y_arr)*0.9, y_arr, alpha=0.15, color=color)
    ax.set_xlabel("ε"); ax.set_ylabel(ylabel)
    ax.set_title(title, color=TEXT_C); ax.grid(alpha=0.3); ax.set_facecolor(PANEL_BG)

eps_ax(fig.add_subplot(gs[0,0]), df_scan.sigma,    "σ (nats)",    ACCENT[1], "Производство энтропии σ(ε)")
eps_ax(fig.add_subplot(gs[0,1]), df_scan.cost,     "C(ε)",        ACCENT[0], "Стоимость транспорта C(ε)")
eps_ax(fig.add_subplot(gs[0,2]), df_scan.entropy,  "H[π]  nats",  ACCENT[2], "Энтропия связывания H[π](ε)")
eps_ax(fig.add_subplot(gs[1,0]), df_scan.n_iters,  "итераций",    ACCENT[4], "Критическое замедление Sinkhorn")
eps_ax(fig.add_subplot(gs[1,1]), df_scan.h_mid,    "H[ρ_{1/2}]",  ACCENT[5], "Энтропия ρ_{t=0.5}(ε)")

# Фазовая диаграмма (C, σ) параметрически по ε
ax_ph = fig.add_subplot(gs[1,2])
sc = ax_ph.scatter(df_scan.cost, df_scan.sigma, c=np.log10(df_scan.epsilon),
                    cmap=CMAP_P, s=130, zorder=5, edgecolors=DARK_BG, lw=1.8)
for _, row in df_scan.iterrows():
    ax_ph.annotate(f"ε={row.epsilon:.2g}", (row.cost, row.sigma),
                    xytext=(5,5), textcoords="offset points",
                    fontsize=7.5, color=TEXT_C, alpha=0.8)
plt.colorbar(sc, ax=ax_ph, label="log₁₀(ε)").ax.yaxis.label.set_color(TEXT_C)
ax_ph.set_xlabel("C  (стоимость)"); ax_ph.set_ylabel("σ  (произв. энтропии)")
ax_ph.set_title("Фазовая диаграмма (C, σ) по ε", color=TEXT_C)
ax_ph.set_facecolor(PANEL_BG)

plt.tight_layout(); plt.show()

In [ ]:
#§6-C ── Интерактивный слайдер ε
def show_bridge_eps(epsilon_idx=4):
    eps = EPSILONS[epsilon_idx]
    x0_v = generate_data(400, 'ring').to(device)
    x1_v = generate_data(400, 'mixture_8').to(device)
    pi_v, _, _, _, _ = sinkhorn(x0_v, x1_v, epsilon=eps, n_iters=200)
    paths_v, _, _ = simulate_bridge(x0_v, x1_v, pi_v, epsilon=eps, n_steps=60)
    regime = ("← детерм. OT" if eps < 0.1 else
              "свободная диффузия →" if eps > 1.0 else "переходный режим")
    fig, axes = plt.subplots(1, 3, figsize=(18, 5), facecolor=DARK_BG)
    fig.suptitle(f"Мост при ε = {eps:.3g}  |  {regime}", color=TEXT_C, fontsize=13)
    for col, (tidx, tlbl) in enumerate(zip([0,30,60],["t=0","t=0.5","t=1"])):
        h  = kde_hist2d(paths_v[tidx].cpu(), bins=60, lim=CFG.lim)
        im = axes[col].imshow(h, origin='lower', extent=[-CFG.lim,CFG.lim]*2,
                               cmap=CMAP_D, aspect='equal', interpolation='bilinear')
        axes[col].set_title(tlbl, color=TEXT_C)
        add_colorbar(axes[col], im); axes[col].set_facecolor(PANEL_BG)
    plt.tight_layout(); plt.show()

if HAS_WIDGETS:
    slider = widgets.IntSlider(value=4, min=0, max=len(EPSILONS)-1, step=1,
                                description='ε-индекс:',
                                style={'description_width':'initial'},
                                layout=widgets.Layout(width='500px'))
    labels = "  ".join([f"[{i}] ε={e:.2g}" for i,e in enumerate(EPSILONS)])
    display(widgets.VBox([
        widgets.HTML(f"<pre style='color:{TEXT_C}'>{labels}</pre>"),
        widgets.interactive(show_bridge_eps, epsilon_idx=slider)
    ]))
else:
    for idx in [1, 4, 7]:
        show_bridge_eps(idx)

print("✓ §6 завершён")

---
## §7  Мост Шрёдингера как динамическая система

Поле дрейфа $b_t(x)$ при фиксированном $t$ — **нестационарное векторное поле**.

**Топологические особенности:**
- *Источники:* $\nabla \cdot b_t > 0$ — области разлёта траекторий
- *Стоки:* $\nabla \cdot b_t < 0$ — области концентрации
- Для класс. броуновского моста: $\nabla \cdot b_t = -d/(1-t)$ — однородный сток

**Показатель Ляпунова** из расходимости соседних траекторий:
$$\lambda_{\rm max} \approx \frac{1}{T}\ln\frac{\|\delta x(T)\|}{\|\delta x(0)\|}$$

$\lambda < 0$ — стягивающий поток к аттрактору $\rho_1$;
$\lambda > 0$ — чувствительность к начальным условиям (хаотическое поведение).

In [ ]:
# §7-A ── Фазовые портреты поля дрейфа
def phase_portrait_ax(ax, drift_fn, t_val, lim=4.0, n_grid=20, title=None):
    xs = torch.linspace(-lim, lim, n_grid)
    ys = torch.linspace(-lim, lim, n_grid)
    GX, GY = torch.meshgrid(xs, ys, indexing='xy')
    grid = torch.stack([GX.reshape(-1), GY.reshape(-1)], 1).to(device)
    t_t  = torch.full((grid.shape[0],), t_val, device=device)
    with torch.no_grad():
        drift = drift_fn(grid, t_t).cpu()
    U = drift[:,0].reshape(n_grid,n_grid).numpy()
    V = drift[:,1].reshape(n_grid,n_grid).numpy()
    speed = np.sqrt(U**2+V**2)
    dUdx  = np.gradient(U, axis=1); dVdy = np.gradient(V, axis=0)
    div   = dUdx + dVdy
    vmax  = np.percentile(np.abs(div), 95)
    im = ax.imshow(div.T, origin='lower', extent=[-lim,lim,-lim,lim],
                   cmap=CMAP_F, alpha=0.5, aspect='equal',
                   norm=Normalize(vmin=-vmax, vmax=vmax))
    ax.streamplot(to_np(xs), to_np(ys), U.T, V.T, color=speed.T,
                   cmap=CMAP_T, linewidth=1.3, density=1.5, arrowsize=1.5)
    ax.set_xlim(-lim,lim); ax.set_ylim(-lim,lim)
    ax.set_facecolor(PANEL_BG)
    if title: ax.set_title(title, color=TEXT_C, fontsize=11)
    return im, div

def neural_drift(x, t): return bridge_net(x, t)

fig, axes = plt.subplots(2, 3, figsize=(19, 12), facecolor=DARK_BG)
fig.suptitle("§7  Фазовые портреты: div(b_t) и поле скоростей",
             color=TEXT_C, fontsize=15, fontweight='bold')

for col, t_val in enumerate([0.2, 0.5, 0.8]):
    im, div = phase_portrait_ax(axes[0,col], neural_drift, t_val,
                                 title=f"Нейрон. мост  t={t_val}")
    # Классический мост из сохранённых дрейфов
    k_dr = min(int(t_val * CFG.n_steps), drifts_c.shape[0]-1)
    pts_t   = paths_c[k_dr].cpu(); drift_t = drifts_c[k_dr].cpu()
    lim_c   = 4.0; ng_c = 16
    gx_c    = torch.linspace(-lim_c,lim_c,ng_c); gy_c = torch.linspace(-lim_c,lim_c,ng_c)
    GX_c, GY_c = torch.meshgrid(gx_c, gy_c, indexing='xy')
    grid_c  = torch.stack([GX_c.reshape(-1), GY_c.reshape(-1)], 1)
    dists_c2 = torch.cdist(grid_c, pts_t)
    nn_c    = dists_c2.topk(min(15, pts_t.shape[0]), largest=False).indices
    dest_c  = torch.stack([drift_t[idx].mean(0) for idx in nn_c])
    U_c = dest_c[:,0].reshape(ng_c,ng_c).numpy()
    V_c = dest_c[:,1].reshape(ng_c,ng_c).numpy()
    spd_c2 = np.sqrt(U_c**2+V_c**2)
    pts_np = to_np(pts_t)
    axes[1,col].scatter(pts_np[::4,0], pts_np[::4,1], s=6, alpha=0.35, color=ACCENT[0], zorder=3)
    axes[1,col].streamplot(to_np(gx_c), to_np(gy_c), U_c.T, V_c.T,
                            color=spd_c2.T, cmap=CMAP_F, density=1.4,
                            linewidth=1.2, arrowsize=1.4)
    axes[1,col].set_xlim(-lim_c,lim_c); axes[1,col].set_ylim(-lim_c,lim_c)
    axes[1,col].set_title(f"Классич. мост  t={t_val}", color=TEXT_C)
    axes[1,col].set_facecolor(PANEL_BG)

plt.tight_layout(); plt.show()

In [ ]:
# §7-B ── Показатели Ляпунова
def lyapunov(paths, dt, n_pairs=60):
    T, N, d = paths.shape
    n_p = min(n_pairs, N//2)
    idx = torch.randperm(N)[:n_p*2].reshape(n_p, 2)
    sep0 = (paths[0, idx[:,0]] - paths[0, idx[:,1]]).norm(dim=-1).clamp_min(1e-8)
    sepT = (paths[-1,idx[:,0]] - paths[-1,idx[:,1]]).norm(dim=-1).clamp_min(1e-8)
    lam  = (torch.log(sepT/sep0) / (T*dt)).mean().item()
    return lam, to_np(sep0), to_np(sepT)

lam_c, s0_c, sT_c = lyapunov(paths_c.cpu(), CFG.dt)
lam_n, s0_n, sT_n = lyapunov(paths_nb.cpu(), CFG.dt)
print(f"λ_max  классич. мост  : {lam_c:+.4f}")
print(f"λ_max  нейрон. мост   : {lam_n:+.4f}")
print("(λ < 0 → стягивающий поток к ρ₁,  λ ≈ 0 → диффузия)")

# Мост к аттрактору Лоренца
print("\nСтроим мост: gaussian → lorenz_xz ...")
x0_lz = generate_data(CFG.n, 'gaussian').to(device)
x1_lz = generate_data(CFG.n, 'lorenz_xz').to(device)
pi_lz, _, _, _, _ = sinkhorn(x0_lz, x1_lz, epsilon=0.12)
paths_lz, _, drifts_lz = simulate_bridge(x0_lz, x1_lz, pi_lz, epsilon=0.12, n_steps=80)
lam_lz, s0_lz, sT_lz = lyapunov(paths_lz.cpu(), 1/80)
print(f"λ_max  мост к Лоренцу : {lam_lz:+.4f}")

# Визуализация
fig, axes = plt.subplots(2, 4, figsize=(22, 10), facecolor=DARK_BG)
fig.suptitle("§7  Показатели Ляпунова и мост к аттрактору Лоренца",
             color=TEXT_C, fontsize=14, fontweight='bold')

def lyap_ax(ax, s0, sT, lam, title):
    ax.scatter(s0, sT, s=20, alpha=0.6, color=ACCENT[0])
    xs_ = np.linspace(s0.min(), s0.max(), 50)
    ax.plot(xs_, xs_*np.exp(lam), '--', color=ACCENT[1], lw=2, label=f"λ={lam:.3f}")
    ax.set_xlabel("δx(0)"); ax.set_ylabel("δx(T)")
    ax.set_title(title, color=TEXT_C); ax.legend(fontsize=9)
    ax.set_facecolor(PANEL_BG)

lyap_ax(axes[0,0], s0_c, sT_c, lam_c, "Ляпунов\nКлассич. мост")
lyap_ax(axes[0,1], s0_n, sT_n, lam_n, "Ляпунов\nНейрон. мост")
lyap_ax(axes[0,2], s0_lz, sT_lz, lam_lz, "Ляпунов\nМост к Лоренцу")

# Аттрактор Лоренца
h_lz = kde_hist2d(x1_lz.cpu(), bins=80, lim=4.0)
im_lz = axes[0,3].imshow(h_lz, origin='lower', extent=[-4,4,-4,4],
                          cmap=CMAP_E, aspect='equal', interpolation='bilinear')
axes[0,3].set_title("Аттрактор Лоренца (x–z)\nρ₁", color=TEXT_C)
add_colorbar(axes[0,3], im_lz); axes[0,3].set_facecolor(PANEL_BG)

# Эволюция к Лоренцу
for col, (tidx, tlbl) in enumerate(zip([0,26,53,80], ["t=0","t=0.33","t=0.67","t=1"])):
    h = kde_hist2d(paths_lz[tidx].cpu(), bins=60, lim=4.0)
    im = axes[1,col].imshow(h, origin='lower', extent=[-4,4,-4,4],
                             cmap=CMAP_D, aspect='equal', interpolation='bilinear')
    axes[1,col].set_title(f"Гаусс→Лоренц  {tlbl}", color=TEXT_C)
    add_colorbar(axes[1,col], im); axes[1,col].set_facecolor(PANEL_BG)

plt.tight_layout(); plt.show()
print("✓ §7 завершён")

---
## §8  Полная картина динамики: анимации и временны́е ряды

In [ ]:
# §8-A ── Анимация эволюции облака частиц
def make_animation(paths, title="Bridge", lim=CFG.lim, n_frames=60, interval=80):
    T_tot = paths.shape[0]; fidx = np.linspace(0, T_tot-1, n_frames, dtype=int)
    paths_np = to_np(paths); N = min(paths_np.shape[1], 400)
    cols_a = np.linspace(0, 1, N)
    fig_a, ax_a = plt.subplots(figsize=(6.5, 6.5), facecolor=DARK_BG)
    ax_a.set_facecolor(PANEL_BG); ax_a.set_xlim(-lim,lim); ax_a.set_ylim(-lim,lim)
    ax_a.set_aspect('equal'); ax_a.grid(alpha=0.18); ax_a.set_title(title, color=TEXT_C, fontsize=12)
    scat = ax_a.scatter([], [], s=9, cmap=CMAP_T, c=[], vmin=0, vmax=1, alpha=0.75)
    txt  = ax_a.text(0.03, 0.96, '', transform=ax_a.transAxes,
                      color=TEXT_C, fontsize=12, va='top', fontweight='bold')
    def init():
        scat.set_offsets(np.zeros((0,2))); txt.set_text(''); return scat, txt
    def upd(frame):
        pts = paths_np[fidx[frame], :N]
        scat.set_offsets(pts); scat.set_array(cols_a)
        txt.set_text(f"t = {fidx[frame]/(T_tot-1):.3f}"); return scat, txt
    anim = animation.FuncAnimation(fig_a, upd, frames=n_frames,
                                    init_func=init, interval=interval, blit=True)
    plt.close(fig_a); return anim

print("Генерируем три анимации ...")
anim_c  = make_animation(paths_c,  "Классический мост: ring → mixture_8")
anim_nb = make_animation(paths_nb, "Нейронный мост:    ring → mixture_8")
anim_lz = make_animation(paths_lz, "Мост к аттрактору Лоренца", lim=4.0)

print("▶  Классический мост:")
display(HTML(anim_c.to_jshtml()))

In [ ]:
print("▶  Нейронный мост:")
display(HTML(anim_nb.to_jshtml()))
print("▶  Мост к аттрактору Лоренца:")
display(HTML(anim_lz.to_jshtml()))

In [ ]:
# §8-B ── Временны́е ряды и пространство-время
def ts_data(paths, drifts, epsilon, n_pts=10):
    ts_v = np.linspace(0,1,n_pts); T = paths.shape[0]
    H_t, KL_t = [], []
    for tv in ts_v:
        k = min(int(tv*(T-1)), T-1); x_k = paths[k].cpu()
        H_t.append(gaussian_entropy(x_k))
        KL_t.append(gaussian_kl(x_k, paths[-1].cpu()))
    _, ep = entropy_production(drifts.cpu(), 1/(T-1), epsilon)
    return np.array(H_t), np.array(KL_t), ep

N_TS2 = 10
H_c2, KL_c2, ep_c2  = ts_data(paths_c,  drifts_c,  CFG.epsilon, N_TS2)
# Для нейр. моста используем конечные-начальные как proxy дрейфа
drifts_nb_proxy = paths_nb[1:] - paths_nb[:-1]
H_n2, KL_n2, ep_n2  = ts_data(paths_nb, drifts_nb_proxy, CFG.epsilon, N_TS2)
H_lz2, KL_lz2, ep_lz2 = ts_data(paths_lz, drifts_lz, 0.12, N_TS2)

ts_ax2 = np.linspace(0,1,N_TS2)

fig = styled_fig(22, 12, "§8  Временны́е ряды физических наблюдаемых")
gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.40, wspace=0.35)

def three_line(ax, ys, colors, labels, ylabel, title):
    for y, c, lbl in zip(ys, colors, labels):
        xs = np.linspace(0,1,len(y))
        ax.plot(xs, y, 'o-', color=c, lw=2, ms=6, label=lbl)
    ax.set_xlabel("t"); ax.set_ylabel(ylabel)
    ax.set_title(title, color=TEXT_C)
    ax.legend(fontsize=8); ax.grid(alpha=0.3); ax.set_facecolor(PANEL_BG)

three_line(fig.add_subplot(gs[0,0]),
           [H_c2, H_n2, H_lz2], ACCENT[:3],
           ["Классич.", "Нейронный", "Лоренц"], "H(t) nats", "Энтропия H(t)")
three_line(fig.add_subplot(gs[0,1]),
           [KL_c2, KL_n2, KL_lz2], ACCENT[:3],
           ["Классич.", "Нейронный", "Лоренц"], "KL(ρ_t‖ρ₁)", "KL к целевому ρ₁")
three_line(fig.add_subplot(gs[0,2]),
           [ep_c2, ep_n2, ep_lz2], ACCENT[:3],
           ["Классич.", "Нейронный", "Лоренц"], "dσ/dt", "Производство энтропии")

# Пространство-время: x₁(t) для классического моста
ax_xt = fig.add_subplot(gs[1,:])
N_XT  = 80
pts_xt = to_np(paths_c[:, :N_XT, 0])
im_xt = ax_xt.imshow(pts_xt.T, origin='lower', extent=[0,1,0,N_XT],
                      aspect='auto', cmap=CMAP_T, interpolation='nearest')
add_colorbar(ax_xt, im_xt, "x₁-координата")
ax_xt.set_xlabel("t (время)"); ax_xt.set_ylabel("Индекс частицы")
ax_xt.set_title("Диаграмма пространство–время: x₁(t) для 80 частиц  (классич. мост)",
                color=TEXT_C)
ax_xt.set_facecolor(PANEL_BG)

plt.tight_layout(); plt.show()
print("✓ §8 завершён")

---
## §9  Сравнительная панель: три метода

| Метод | Граничные условия | Нейросеть? | Механизм |
|---|---|---|---|
| **Диффузия VE-SDE** | $\rho_{\rm noise} \to \rho_1$ | ✓ ScoreNet | Score matching |
| **Классич. мост** | $\rho_0 \to \rho_1$ | ✗ | Sinkhorn + SDE |
| **Нейрон. мост (FM)** | $\rho_0 \to \rho_1$ | ✓ BridgeDriftNet | Flow Matching |

In [ ]:
import numpy as np
import torch

def bootstrap_metric(metric_func, x, y, n_boot=100, sample_frac=0.8, **kwargs):
    """
    Оценка метрики с погрешностью методом бутстрепа.

    Parameters:
    -----------
    metric_func : callable
        Функция метрики, принимающая два тензора (x, y) и возвращающая скаляр.
    x, y : torch.Tensor
        Исходные выборки.
    n_boot : int
        Число бутстреп-итераций.
    sample_frac : float
        Доля выборки для каждого ресэмплинга (по умолчанию 0.8).
    **kwargs : dict
        Дополнительные аргументы для metric_func.

    Returns:
    --------
    mean_val, std_val : float
        Среднее значение метрики и её стандартное отклонение.
    """
    values = []
    n = min(len(x), len(y))
    n_sample = int(n * sample_frac)

    for _ in range(n_boot):
        # Ресэмплинг с возвращением
        idx_x = torch.randint(0, n, (n_sample,), device=x.device)
        idx_y = torch.randint(0, n, (n_sample,), device=y.device)
        x_s = x[idx_x]
        y_s = y[idx_y]

        val = metric_func(x_s, y_s, **kwargs)
        values.append(val)

    values = np.array(values)
    return values.mean(), values.std(ddof=1)

# Модифицированные версии метрик (возвращают float, а не строку)
def swd(x, y, n_proj=None):
    n_proj = n_proj or CFG.n_proj
    x, y = x.to(device), y.to(device)
    dirs = F.normalize(torch.randn(n_proj, x.shape[1], device=device), dim=1)
    xs,_ = (x@dirs.T).sort(0); ys,_ = (y@dirs.T).sort(0)
    return (xs-ys).abs().mean().item()

def mmd(x, y):
    x, y = x.to(device), y.to(device)
    dxy  = torch.cdist(x,y).pow(2)
    dxx  = torch.cdist(x,x).pow(2)
    dyy  = torch.cdist(y,y).pow(2)
    sig  = dxy.median().clamp_min(1e-8).sqrt()*0.5
    g    = 1/(2*sig**2+1e-12)
    return ((-g*dxx).exp().mean()+(-g*dyy).exp().mean()-2*(-g*dxy).exp().mean()).item()

# --- Основная часть: вычисление метрик с погрешностью ---
N_CMP = 1000
ref_tgt = generate_data(N_CMP, 'mixture_8').to(device)

final_diff   = sample_diffusion_ode(score_net, N_CMP, n_steps=400)
final_bridge = paths_c[-1, :N_CMP].to(device)
final_neural, _ = sample_bridge_ode(bridge_net, 'ring', n=N_CMP)

methods = {
    "Диффузия (VE-SDE)": final_diff,
    "Классич. мост":     final_bridge,
    "Нейрон. мост (FM)": final_neural,
}

rows = []
for name, samp in methods.items():
    s = samp[:N_CMP].to(device)

    # Вычисление метрик с погрешностью через бутстреп
    swd_val, swd_err = bootstrap_metric(swd, s, ref_tgt, n_boot=100)
    mmd_val, mmd_err = bootstrap_metric(mmd, s, ref_tgt, n_boot=100)

    # Для KL и энтропии бутстреп может быть нестабилен из-за оценки плотности;
    # если нужно — можно добавить аналогично, но с осторожностью
    kl_val = gaussian_kl(s.cpu(), ref_tgt.cpu())  # без ошибки, или добавить бутстреп
    h_val  = gaussian_entropy(s.cpu())

    rows.append({
        "Метод": name,
        "SWD":  f"{swd_val:.4f} ± {swd_err:.4f}",
        "MMD":  f"{mmd_val:.4f} ± {mmd_err:.4f}",
        "KL":   f"{kl_val:.4f}",
        "H":    f"{h_val:.4f}"
    })

df_cmp = pd.DataFrame(rows)
print("╔" + "═"*66 + "╗")
print("║            СРАВНИТЕЛЬНЫЕ МЕТРИКИ (ring → mixture_8)             ║")
print("╠" + "═"*66 + "╣")
header = f"{'Метод':<26} {'SWD':>8} {'MMD':>10} {'KL':>9} {'H':>9}"
print(f"║  {header}  ║")
print("╠" + "─"*66 + "╣")
for _, r in df_cmp.iterrows():
    row_str = f"{r['Метод']:<26} {r['SWD']:>8} {r['MMD']:>10} {r['KL']:>9} {r['H']:>9}"
    print(f"║  {row_str}  ║")
print("╚" + "═"*66 + "╝")

In [ ]:
# §9 ── Визуальная сетка 3×4
fig = styled_fig(24, 17, "§9  Сравнительная панель: все три метода")
gs  = gridspec.GridSpec(3, 4, figure=fig, hspace=0.30, wspace=0.22)

row_configs = [
    ("Диффузия (VE-SDE)",    None,          final_diff,    ACCENT[0]),
    ("Классич. мост",         paths_c.cpu(), final_bridge,  ACCENT[1]),
    ("Нейрон. мост (FM)",    paths_nb.cpu(), final_neural,  ACCENT[2]),
]
t_inds2 = [0, 40, 80, 120]; t_lbls2 = ["t=0","t=0.33","t=0.67","t=1"]

for row, (rname, rpaths, rfinal, rcol) in enumerate(row_configs):
    for col, (tidx, tlbl) in enumerate(zip(t_inds2, t_lbls2)):
        ax = fig.add_subplot(gs[row, col])
        if rpaths is not None and tidx < rpaths.shape[0]:
            xplt = rpaths[tidx]
        elif col == 0 and row == 0:
            xplt = generate_data(N_CMP, 'gaussian')
        elif col == 3:
            xplt = rfinal.cpu()
        else:
            ax.set_facecolor(PANEL_BG)
            ax.text(0.5,0.5,"—",ha='center',va='center',
                    transform=ax.transAxes,color=TEXT_C,fontsize=24)
            if col==0: ax.set_ylabel(rname, color=rcol, fontsize=10, fontweight='bold')
            if row==0: ax.set_title(tlbl, color=TEXT_C, fontsize=11)
            continue

        lim_p = 5.0
        h  = kde_hist2d(xplt, bins=60, lim=lim_p)
        im = ax.imshow(h, origin='lower', extent=[-lim_p,lim_p,-lim_p,lim_p],
                        cmap=CMAP_D, aspect='equal', interpolation='bilinear')
        if col==0: ax.set_ylabel(rname, color=rcol, fontsize=10, fontweight='bold')
        if row==0: ax.set_title(tlbl, color=TEXT_C, fontsize=11)
        ax.set_xticks([]); ax.set_yticks([]); ax.set_facecolor(PANEL_BG)
        for sp in ax.spines.values(): sp.set_edgecolor(rcol); sp.set_linewidth(2.0)

plt.tight_layout(); plt.show()
print("✓ §9 завершён")

---
## §10  Физическая интерпретация и таблица аналогий

### Унифицирующая таблица: четыре языка одной теории

| Объект | Классич. механика | Стохаст. термодинамика | Квантовая механика | Мост Шрёдингера |
|---|---|---|---|---|
| Поле | Потенциал $V(x)$ | Свободная энергия $F$ | Волновая функция $\psi$ | Дрейф $b_t(x)$ |
| Уравнение | Гамильтона–Якоби | Ланжевена | Шрёдингера | SDE моста |
| Принцип | Наим. действия | Мин. σ | Мин. неопред-ти | EOT (Sinkhorn) |
| Инверсия t | $t \to -t$ | KL(P→ ‖ P←) = σ | $\psi \to \psi^*$ | Обращённый мост |
| Равновесие | Неподвижная точка | $\rho\propto e^{-V/T}$ | Основное состояние | $\varepsilon \to \infty$ |

### Открытые вопросы для исследования

1. **Критический ε и фазовый переход:** существует ли $\varepsilon_c$, при котором
   структура $\pi^*_\varepsilon$ меняется топологически? Связан ли он с геометрией $(\rho_0, \rho_1)$?

2. **Хаос и производство энтропии:** для хаотических аттракторов ($\lambda > 0$) —
   существует ли аналог теоремы Крукса, связывающий $\lambda$ и $\sigma$?

3. **Квантовый мост:** замена $\sqrt{\varepsilon}dW \to \sqrt{\hbar}dW$, переход к мнимому
   времени $t \to it$ — уравнение Нельсона. Можно ли имитировать квантовые переходы через мост?

4. **Неравновесный стационарный цикл:** если $\rho_0 = \rho_1$ но траектории нетривиальны —
   возникает ли ненулевой вероятностный ток? Аналог неравновесного стационарного состояния.

5. **Высокие измерения:** как масштабируются $\sigma(\varepsilon)$, $\lambda$, фазовая
   диаграмма при $d: 2 \to 100$?

In [ ]:
# §10-B ── Финальный отчёт
print()
print("╔" + "═"*62 + "╗")
print("║     SCHRÖDINGER BRIDGE RESEARCH PLATFORM — ОТЧЁТ          ║")
print("╠" + "═"*62 + "╣")
print(f"║  n_samples   = {CFG.n:<8d}  epsilon = {CFG.epsilon:<8.3f}               ║")
print(f"║  n_steps     = {CFG.n_steps:<8d}  hidden  = {CFG.hidden:<8d}               ║")
print(f"║  epochs_diff = {CFG.epochs_diff:<8d}  device  = {str(device):<12s}             ║")
print("╠" + "═"*62 + "╣")
print("║  Термодинамика классического моста (ring → mixture_8)      ║")
print(f"║    H[ρ₀]      = {entropies[0]:>8.4f} nats                            ║")
print(f"║    H[ρ₁]      = {entropies[-1]:>8.4f} nats                            ║")
print(f"║    σ_total    = {sigma_total:>8.4f} nats                            ║")
print(f"║    λ_Lyapunov = {lam_c:>+8.4f}                                 ║")
print(f"║    Стрела вр. = {ta:>8.4f}                                 ║")
print("╠" + "═"*62 + "╣")
print("║  Качество транспорта                                       ║")
for r in rows:
    print(f"║  {r['Метод']:<26}: SWD={r['SWD']}  MMD={r['MMD']}       ║")
print("╠" + "═"*62 + "╣")
print("║  ε-скан: {:.4g} ≤ ε ≤ {:.4g}  ({} точек)".format(
    min(EPSILONS), max(EPSILONS), len(EPSILONS)).ljust(62) + "║")
print("╚" + "═"*62 + "╝")
print()
print("Платформа готова. Для углублённого анализа:")
print("  · увеличьте CFG.n, CFG.epochs_diff, CFG.epochs_bridge")
print("  · добавьте новые пары (src, tgt) в §2 / §7")
print("  · расширьте EPSILONS в §6 для более тонкого ε-скана");